In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

import networkx as nx
import matplotlib.pyplot as plt

from residual_correlation_graph import build_residual_correlation_graph
from residual_regression_correlations_with_calandar_features import build_residual_regression_correlation_graph

In [2]:
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['item_id'] = df['item_id'].astype(int) # or .astype(str)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)

m0 = df[DATE_COL].dt.month - 1
df["month_sin"] = np.sin(2*np.pi*m0 / 12)
df["month_cos"] = np.cos(2*np.pi*m0 / 12)

df["doy"] = df[DATE_COL].dt.dayofyear # 1..365/366
doy0 = df["doy"] - 1
P = 366 # safe; or use 365 if you drop leap years
df["doy_sin"] = np.sin(2*np.pi*doy0 / P)
df["doy_cos"] = np.cos(2*np.pi*doy0 / P)

promo_types= [col for col in df.columns if col.startswith("promo_type_")] # Binary columns indicating presence of specific promotion types
promo_values= [col for col in df.columns if col.startswith("promo_value_")] # Numerical columns indicating the value of specific promotion types

calendar_cols = ["day_of_week", "day_of_month", "moy", "doy", "is_weekend"]
calendar_trigonometric_cols = ["month_sin", "month_cos", "doy_sin", "doy_cos"]
categorical_cols = ["cat_label", "sdep_label", "dept_label"]

df = df.sort_values(["item_id", DATE_COL]).reset_index(drop=True)

df

Loading data from ../dataset/data_andre.feather...
1082371


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,store_id,day_of_week,day_of_month,moy,doy,is_weekend,month_sin,month_cos,doy_sin,doy_cos
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,6269,5,23,0,23,1,0.0,1.000000,0.368763,0.929523
1,2021-01-24,27,14,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,6269,6,24,0,24,1,0.0,1.000000,0.384665,0.923056
2,2021-01-25,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,6269,0,25,0,25,0,0.0,1.000000,0.400454,0.916317
3,2021-01-26,27,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,6269,1,26,0,26,0,0.0,1.000000,0.416125,0.909308
4,2021-01-27,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,6269,2,27,0,27,0,0.0,1.000000,0.431673,0.902030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082366,2023-02-18,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,6269,5,18,1,49,1,0.5,0.866025,0.733885,0.679273
1082367,2023-02-19,900087600,46,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,6269,6,19,1,50,1,0.5,0.866025,0.745438,0.666575
1082368,2023-02-20,900087600,28,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,6269,0,20,1,51,0,0.5,0.866025,0.756771,0.653680
1082369,2023-02-21,900087600,31,bottled water,pos subd grocery other,pos dept grocery,water/isotonics,0,0.0,0,...,6269,1,21,1,52,0,0.5,0.866025,0.767880,0.640593


In [3]:
# Check for duplicated dates within each item_id
duplicates = df[df.duplicated(subset=['item_id', DATE_COL], keep=False)]

if duplicates.empty:
    print("No duplicated dates found for any item_id.")
else:
    print(f"Found {len(duplicates)} rows with duplicated dates per item_id:")
    print(duplicates.sort_values(['item_id', DATE_COL]))

No duplicated dates found for any item_id.


In [4]:
'''
adj_list, corr_matrix= build_residual_correlation_graph(df, date_col=DATE_COL, target_col=TARGET_COL, corr_threshold=0.8)

# print adjacent pairs above threshold, and their correlation values
print("Adjacent pairs above threshold:")
printed_pairs = set()

for node, neighbors in adj_list.items():
    for neighbor in neighbors:
        # Create a sorted tuple to track uniqueness (since it's an undirected graph)
        pair = tuple(sorted([node, neighbor]))
        if pair not in printed_pairs:
            corr_val = corr_matrix.loc[node, neighbor]
            print(f"Item {node} - Item {neighbor}: Correlation = {corr_val:.4f}")
            printed_pairs.add(pair)
'''

'\nadj_list, corr_matrix= build_residual_correlation_graph(df, date_col=DATE_COL, target_col=TARGET_COL, corr_threshold=0.8)\n\n# print adjacent pairs above threshold, and their correlation values\nprint("Adjacent pairs above threshold:")\nprinted_pairs = set()\n\nfor node, neighbors in adj_list.items():\n    for neighbor in neighbors:\n        # Create a sorted tuple to track uniqueness (since it\'s an undirected graph)\n        pair = tuple(sorted([node, neighbor]))\n        if pair not in printed_pairs:\n            corr_val = corr_matrix.loc[node, neighbor]\n            print(f"Item {node} - Item {neighbor}: Correlation = {corr_val:.4f}")\n            printed_pairs.add(pair)\n'

In [5]:
'''
def build_residual_regression_correlation_graph(
    df: pd.DataFrame,
    node_col: str = "item_id",
    date_col: str = "date",
    value_col: str = "value",
    feature_cols: list[str] | None = None,
    categorical_cols: list[str] | None = None,
    fit_mode: str = "full_sample",
    min_train_size: int = 28,
    drop_first: bool = True,
    min_overlap: int = 10,
    corr_method: str = "pearson",
    corr_threshold: float = 0.2,
    absolute_corr: bool = True,
):
'''
adj_list, corr_matrix= build_residual_regression_correlation_graph(
    df,
    node_col="item_id",
    date_col=DATE_COL,
    value_col=TARGET_COL,
    feature_cols=calendar_trigonometric_cols + promo_types + promo_values,
    categorical_cols=categorical_cols,
    fit_mode="full_sample",
    min_train_size=28,
    drop_first=True,
    min_overlap=10,
    corr_method="pearson",
    corr_threshold=0.2,
    absolute_corr=True,
)

In [8]:
print("Adjacent pairs above threshold:")
printed_pairs = set()

# print top 20 pairs by absolute correlation value
corr_pairs = []
for node, neighbors in adj_list.items():
    for neighbor in neighbors:
        # Create a sorted tuple to track uniqueness (since it's an undirected graph)
        pair = tuple(sorted([node, neighbor]))
        if pair not in printed_pairs:
            corr_val = corr_matrix.loc[node, neighbor]
            corr_pairs.append((pair[0], pair[1], corr_val))
            printed_pairs.add(pair)
# Sort by absolute correlation value and print top 20
corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print("Top 20 pairs by absolute correlation value:")
for item1, item2, corr_val in corr_pairs[:20]:
    print(f"Item {item1} - Item {item2}: Correlation = {corr_val:.4f}")

Adjacent pairs above threshold:
Top 20 pairs by absolute correlation value:
Item 154813 - Item 154814: Correlation = 0.8155
Item 26008 - Item 26924: Correlation = 0.6911
Item 26002 - Item 26012: Correlation = 0.6529
Item 907969 - Item 911753: Correlation = 0.6449
Item 213629 - Item 911753: Correlation = 0.6343
Item 901555 - Item 930471: Correlation = 0.6275
Item 213628 - Item 911753: Correlation = 0.6270
Item 213625 - Item 213628: Correlation = 0.6247
Item 213625 - Item 213629: Correlation = 0.6160
Item 213627 - Item 907969: Correlation = 0.6129
Item 18605 - Item 18607: Correlation = 0.6110
Item 904112 - Item 919137: Correlation = 0.6103
Item 213628 - Item 213629: Correlation = 0.6073
Item 213627 - Item 213629: Correlation = 0.5956
Item 213627 - Item 911753: Correlation = 0.5935
Item 26768 - Item 26924: Correlation = 0.5915
Item 8513 - Item 26924: Correlation = 0.5901
Item 213625 - Item 213627: Correlation = 0.5895
Item 213628 - Item 907969: Correlation = 0.5849
Item 26702 - Item 91175